# Trabalho 2 – Algoritmos Genéticos: Caixeiro Viajante

**Integrantes:** Fabian Formento, Lucas Visconti, Cauã F. Bertolini, Luiz C. Martendal, Eduarda D. Mazureck

Objetivo: encontrar, com um algoritmo genético, a rota mais curta que visita 20 cidades exatamente uma vez e volta à cidade inicial.

In [1]:
import numpy as np  # cálculos vetoriais e números aleatórios

## 1. Função de aptidão (fitness)

### 1.1 Carregando as cidades
O arquivo `cidades.mat` é texto puro, por isso usamos `np.loadtxt` (o `scipy.io.loadmat` não consegue lê-lo).
A linha 0 contém as coordenadas `x` e a linha 1 as coordenadas `y`.

In [2]:
# Carrega as coordenadas: matriz 2 x 20
x, y = np.loadtxt('cidades.mat')

Ncidades = len(x)  # número de cidades (20)
print('Número de cidades:', Ncidades)

Número de cidades: 20


### 1.2 Matriz de distâncias
Calculamos uma única vez a distância euclidiana entre todos os pares de cidades:

$$d(c_i, c_j) = \sqrt{(x_i - x_j)^2 + (y_i - y_j)^2}$$

A matriz é simétrica, pois $d(c_i, c_j) = d(c_j, c_i)$.

In [3]:
def matriz_distancias(x, y):
    # Diferenças entre todas as coordenadas (broadcasting gera matrizes n x n)
    dx = x[:, None] - x[None, :]
    dy = y[:, None] - y[None, :]
    # Distância euclidiana entre cada par de cidades
    return np.sqrt(dx**2 + dy**2)

dcidade = matriz_distancias(x, y)  # matriz 20 x 20

### 1.3 Função de aptidão
Cada cromossomo é uma permutação das cidades. Internamente as cidades são numeradas de **0 a 19** (índices do Python);
na exibição dos resultados somamos 1 para ficar de 1 a 20, como no enunciado.

A aptidão é a distância total do percurso, incluindo a volta à cidade inicial ($c_{n+1} = c_1$):

$$D = \sum_{k=1}^{n} d(c_k, c_{k+1})$$

Quanto **menor** o valor de $D$, **mais apto** é o cromossomo.

In [4]:
def aptidao(pop, dcidade=dcidade):
    # Garante formato 2D: (Npop, Ncidades), mesmo se vier um único cromossomo
    pop = np.atleast_2d(pop)
    # Acrescenta a primeira cidade no final de cada rota (volta ao início),
    # equivalente ao tour = [pop pop(:,1)] do Matlab
    tour = np.hstack([pop, pop[:, :1]])
    # Distância de cada trecho c_k -> c_k+1 de cada cromossomo
    trechos = dcidade[tour[:, :-1], tour[:, 1:]]
    # Soma os trechos: distância total de cada cromossomo
    return trechos.sum(axis=1)

### 1.4 Testes da função de aptidão

In [5]:
# Teste 1: quatro cidades nos cantos de um quadrado 1 x 1
xq = np.array([0, 1, 1, 0])
yq = np.array([0, 0, 1, 1])
dq = matriz_distancias(xq, yq)
assert np.isclose(aptidao([0, 1, 2, 3], dq)[0], 4.0)                # contorno do quadrado = 4
assert np.isclose(aptidao([0, 2, 1, 3], dq)[0], 2 + 2*np.sqrt(2))  # rota cruzando as diagonais

# Teste 2: comparação com a versão em laços (igual ao código Matlab da professora)
def aptidao_lacos(pop):
    Npop, Nc = pop.shape
    tour = np.hstack([pop, pop[:, :1]])
    dist = np.zeros(Npop)
    for i in range(Npop):
        for j in range(Nc):
            dist[i] += dcidade[tour[i, j], tour[i, j + 1]]
    return dist

rng = np.random.default_rng(0)
pop_teste = np.array([rng.permutation(Ncidades) for _ in range(20)])  # 20 rotas aleatórias
assert np.allclose(aptidao(pop_teste), aptidao_lacos(pop_teste))

# Teste 3: rotacionar ou inverter a rota não muda a distância
rota = pop_teste[0]
assert np.isclose(aptidao(rota)[0], aptidao(np.roll(rota, 5))[0])
assert np.isclose(aptidao(rota)[0], aptidao(rota[::-1])[0])

# Teste 4: a matriz de distâncias é simétrica e tem diagonal zero
assert np.allclose(dcidade, dcidade.T)
assert np.allclose(np.diag(dcidade), 0)

print('Todos os testes passaram!')
print('Aptidão das 5 primeiras rotas aleatórias:', np.round(aptidao(pop_teste)[:5], 4))

Todos os testes passaram!
Aptidão das 5 primeiras rotas aleatórias: [ 9.7768 12.2454  8.8427 10.218  10.5802]
